# Customer Classification - Training Pipeline

This notebook demonstrates the complete training pipeline for the customer classification project. You can use this for exploratory data analysis and understanding the workflow before running the automated training script.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src directory to path
sys.path.append('../src')

# Import project modules
from data_preprocessing import DataPreprocessor
from feature_engineering import FeatureEngineer
from train_model import ModelTrainer

# Set up visualization
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Data Loading and Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Load and preprocess data
print("Starting data preprocessing...")
processed_data, data_path = preprocessor.preprocess_pipeline()

print(f"\nProcessed data shape: {processed_data.shape}")
print(f"\nColumns: {list(processed_data.columns)}")
print(f"\nData saved to: {data_path}")

In [ ]:
# Display basic statistics
print("Basic Statistics:")
display(processed_data.describe())

# Display first few rows
print("\nFirst 5 rows:")
display(processed_data.head())

## 2. Exploratory Data Analysis

In [ ]:
# Customer segment distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
segment_counts = processed_data['CustomerSegment_Encoded'].value_counts()
segment_labels = ['Low Value', 'Medium Value', 'High Value']
plt.pie(segment_counts.values, labels=segment_labels, autopct='%1.1f%%')
plt.title('Customer Segment Distribution')

plt.subplot(1, 2, 2)
sns.countplot(data=processed_data, x='CustomerSegment_Encoded')
plt.xticks([0, 1, 2], segment_labels)
plt.title('Customer Segment Counts')
plt.xlabel('Customer Segment')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by segment
numeric_features = ['TotalAmount_Sum', 'TransactionCount', 'UniqueProducts', 'CustomerTenureDays']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(numeric_features):
    if feature in processed_data.columns:
        sns.boxplot(data=processed_data, x='CustomerSegment_Encoded', y=feature, ax=axes[i])
        axes[i].set_xticklabels(['Low', 'Medium', 'High'])
        axes[i].set_title(f'{feature} by Segment')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
numeric_cols = processed_data.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['CustomerID', 'CustomerSegment_Encoded']]

correlation_matrix = processed_data[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

## 3. Feature Engineering

In [ ]:
# Initialize feature engineer
fe = FeatureEngineer()

# Apply feature engineering
print("Starting feature engineering...")
X_final, y_final = fe.feature_engineering_pipeline(processed_data)

print(f"\nFinal feature set shape: {X_final.shape}")
print(f"Target variable shape: {y_final.shape}")
print(f"Selected features: {fe.selected_features}")

In [ ]:
# Display feature importance from feature selection
if hasattr(fe.feature_selector, 'scores_'):
    feature_scores = pd.DataFrame({
        'Feature': fe.selected_features,
        'Score': fe.feature_selector.scores_
    }).sort_values('Score', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_scores.head(10), x='Score', y='Feature')
    plt.title('Top 10 Feature Scores')
    plt.show()
    
    display(feature_scores.head(10))

## 4. Model Training and Evaluation

In [ ]:
# Initialize model trainer
trainer = ModelTrainer()

# Prepare data
X, y = trainer.prepare_data()

# Split data
X_train, X_test, y_train, y_test = trainer.split_data(X, y)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Initialize and train models
trainer.initialize_models()
trainer.train_models(X_train, y_train)

# Evaluate models
trainer.evaluate_models(X_test, y_test)

# Select best model
best_name, best_scores = trainer.select_best_model()

print(f"\nBest model: {best_name}")
print(f"Best F1 Score: {best_scores['f1_score']:.4f}")
print(f"Best Accuracy: {best_scores['accuracy']:.4f}")

In [ ]:
# Create comparison plots
plot_path = trainer.create_comparison_plots()
print(f"Model comparison plot saved to: {plot_path}")

# Display the plot
from IPython.display import Image
display(Image(filename=plot_path))

## 5. Model Analysis

In [ ]:
# Feature importance for the best model
if hasattr(trainer.best_model, 'feature_importances_'):
    importance_dict = dict(zip(fe.selected_features, trainer.best_model.feature_importances_))
    importance_df = pd.DataFrame(
        list(importance_dict.items()),
        columns=['Feature', 'Importance']
    ).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=importance_df.head(10), x='Importance', y='Feature')
    plt.title('Top 10 Feature Importances')
    plt.show()
    
    display(importance_df.head(10))

In [ ]:
# Confusion matrix for the best model
from sklearn.metrics import confusion_matrix, classification_report

y_pred = trainer.best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {trainer.best_model_name}')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))

## 6. Save Model and Artifacts

In [ ]:
# Save the best model and artifacts
model_path = trainer.save_model()
print(f"Model saved to: {model_path}")

print("\nAll artifacts saved successfully!")
print("You can now run the Streamlit app with: streamlit run app/app.py")

## 7. Test Prediction Pipeline

In [ ]:
# Test the prediction pipeline
from predict import Predictor

predictor = Predictor()
predictor.load_artifacts()

# Create sample input
sample_input = {
    'TotalAmount_Sum': 1500.0,
    'TotalAmount_Mean': 75.0,
    'TransactionCount': 20,
    'Quantity_Sum': 150,
    'Quantity_Mean': 7.5,
    'Price_Mean': 10.0,
    'Country_Encoded': 100,
    'UniqueProducts': 18,
    'CustomerTenureDays': 400,
    'AvgDaysBetweenPurchases': 20,
    'FirstPurchase_Year': 2023,
    'FirstPurchase_Month': 1,
    'LastPurchase_Year': 2023,
    'LastPurchase_Month': 12
}

# Make prediction
result = predictor.predict(sample_input)

print("Prediction Result:")
print(f"Predicted Segment: {result['segment']}")
print(f"Prediction (numeric): {result['prediction']}")
if result['probabilities']:
    print(f"Probabilities: {result['probabilities']}")

## Summary

This notebook demonstrates the complete ML pipeline:

1. ✅ **Data Preprocessing**: Cleaned and processed raw transaction data
2. ✅ **Feature Engineering**: Created customer-level features and selected important ones
3. ✅ **Model Training**: Trained and evaluated 3 different ML models
4. ✅ **Model Selection**: Selected the best performing model
5. ✅ **Model Persistence**: Saved model and preprocessing artifacts
6. ✅ **Inference Testing**: Verified the prediction pipeline works

The project is now ready for use with the Streamlit application!